# 6. Longitudinal Analysis

In this notebook, we examine how the infant gut microbiome changes over time by following children across multiple sampling points. We begin by preparing the data for longitudinal analysis, which requires creating a properly structured dataframe where each sample is linked to its subject ID, age, covariates, and the selected diversity metric. After this, we generate an additional dataframe that captures how much the microbiome changes between visits. This is done using first differences, which measure how an alpha-diversity value changes from one timepoint to the next. These derived data help us quantify both the direction and the magnitude of microbiome change over time.

Using this prepared data, we then apply several longitudinal tools. Volatility plots show how diversity (or its changes) develops over time for each child, giving a clear picture of individual trajectories. Feature volatility identifies which bacterial taxa change the most with age, helping us understand which microbes are important during gut microbiome maturation. Finally, Linear Mixed-Effects Models (LME) allow us to test whether factors such as age, diet, or geography have a significant influence on microbiome development while accounting for repeated measurements from the same child.

### Notebook Structure

**0.** Setup  

**1.** Longitudinal data preparation  
&nbsp;&nbsp;&nbsp;&nbsp;**1.1** Link alpha diversity metrics to metadata  
&nbsp;&nbsp;&nbsp;&nbsp;**1.2** Calculate first differences between consecutive timepoints  
&nbsp;&nbsp;&nbsp;&nbsp;**1.3** Calculate first distances in beta diversity over time 

**2.** Longitudinal exploration of diversity patterns  

**3.** Longitudinal modeling of alpha diversity  
&nbsp;&nbsp;&nbsp;&nbsp;**3.1** Define function for linear mixed-effect modeling   
&nbsp;&nbsp;&nbsp;&nbsp;**3.2** Fit linear mixed-effects models for alpha diversity    
&nbsp;&nbsp;&nbsp;&nbsp;**3.3** Summarize and interpret model results  

**4.** Longitudinal modeling of taxa associated with microbiome development  
&nbsp;&nbsp;&nbsp;&nbsp;**4.1** Identify age- or diet-associated taxa using feature volatility  
&nbsp;&nbsp;&nbsp;&nbsp;**4.2** Fit linear mixed-effects models for selected taxa  
&nbsp;&nbsp;&nbsp;&nbsp;**4.3** Summarize and interpret model results  



<div style="border: 2px solid #1f77b4; padding: 10px; border-radius: 6px; background-color: #eaf3fb;">
<b>Research questions this notebook answers:</b>
    
- How does alpha diversity (e.g., richness, Shannon diversity) vary across metadata groups, including age, feeding pattern, delivery mode, treatment, sex, and geographic location?
    
- Do metadata factors such as feeding pattern, delivery mode, treatment, sex, and geographic location influence the trajectory of alpha diversity change over time?
    
- Which microbial features (taxa) show the strongest age-related changes, and how do their longitudinal patterns differ across metadata factors such as geographic location, delivery mode, or treatment exposure?
    
</div>


## 0. Setup

In [1]:
# Import all necessary packages
import os
import glob
import shutil
import zipfile
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import qiime2 as q2
from qiime2 import Visualization
import IPython
from ipywidgets import Dropdown, VBox
from functools import reduce
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

%matplotlib inline

In [2]:
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts").


In [3]:
# Data directories
raw_data_dir = "../data/raw"
processed_data_dir = "../data/processed"

meta_data_dir = "../data/processed/metadata"
denoising_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"
phylogeny_data_dir = "../data/processed/phylogeny"
diversity_data_dir = "../data/processed/diversity"
longitudinal_data_dir = "../data/processed/longitudinal"

# Create directories
!mkdir -p $raw_data_dir $processed_data_dir $meta_data_dir \
         $denoising_data_dir $taxonomy_data_dir $phylogeny_data_dir \
         $diversity_data_dir $longitudinal_data_dir

In [4]:
%%bash -s "$longitudinal_data_dir"
mkdir -p "$1"

### Set run and metric to work with

This menu lets you select the run folder to analyse, as well as the alpha diversity metric and the time variable.

The run selector lists all processed datasets available in `../data/processed`.  
For longitudinal analysis, you can choose whether time is represented in months or days.

These settings control how the data are loaded and processed in the analysis cells below.

By default, all options match the settings used for the main project results.  
The widgets are included mainly as a convenient way to rerun the analysis with different settings when exploring alternative comparisons.


In [5]:
# Collect available runs
run_options = sorted([
    d for d in os.listdir(diversity_data_dir)
    if os.path.isdir(os.path.join(diversity_data_dir, d))
])

# Dropdown: which run?
run_selector = Dropdown(
    options=run_options,
    value = "depth-14000_no-filter",
    description="Run:",
)

# Dropdown: which time variable?
time_selector = Dropdown(
    options=["age_months", "age_days"],
    value="age_months",
    description="Time column:",
)

# Display all widgets
display(VBox([run_selector, time_selector]))

# Save final selections into variables
run = run_selector.value
time_column = time_selector.value

## 1. Longitudinal data preparation  

### 1.1 Link alpha diversity metrics to metadata

Here, we export the alpha diversity `.qza` file, convert it to a `.tsv` file,
and merge it with the sample metadata.

This results in a single table that contains:
- the sample ID  
- all sample-level metadata  
- the alpha diversity value for each sample  

Many QIIME longitudinal tools require this combined metadata table because
they operate on metadata columns rather than directly on QIIME artifacts.
By storing diversity values as metadata, QIIME can treat them in the same
way as time variables or grouping factors when performing longitudinal
visualization and analysis [[National Cancer Institute, QIIME 2 Lesson 5](https://bioinformatics.ccr.cancer.gov/docs/qiime2/Lesson5/)].. First, we set file paths to keep the rest of the analysis clean.

In [6]:
# Define the main analysis directory for this run
analysis_dir = f"{longitudinal_data_dir}/{run}"

# Subdirectories for different parts of the longitudinal analysis
alpha_raw_dir = f"{analysis_dir}/alpha_raw"                 # alpha diversity values linked to metadata
alpha_lme_results_dir = f"{analysis_dir}/alpha_lme_results" # results from linear mixed-effects models
alpha_fd_dir = f"{analysis_dir}/alpha_firstdiff"            # first differences of alpha diversity
beta_fd_dir = f"{analysis_dir}/beta_firstdist"              # first distances of beta diversity
volatility_raw_dir = f"{analysis_dir}/volatility_raw"       # volatility plots for raw diversity
volatility_fd_dir = f"{analysis_dir}/volatility_firstdiff"  # volatility plots for first differences / distances

!mkdir -p {alpha_raw_dir} {alpha_fd_dir} {volatility_raw_dir} {volatility_fd_dir} {beta_fd_dir} {alpha_lme_results_dir}

In this step, we load all alpha diversity metrics that were calculated during preprocessing
and link them to the sample metadata.

Each alpha diversity metric is stored as a separate QIIME `.qza` file.
We export these files to `.tsv` format, clean the column names,
and store them in a list so they can be merged in the next step.

In [7]:
# Directory containing alpha diversity metric QZA files
alpha_metrics_dir = f"{diversity_data_dir}/{run}/alpha/metrics"

# Find all alpha diversity vector files
alpha_qza_files = glob.glob(f"{alpha_metrics_dir}/*_vector.qza")

# Load sample metadata
metadata = pd.read_csv(f"{meta_data_dir}/metadata_merged.tsv", sep="\t")

# List to store alpha diversity dataframes
alpha_dfs = []

for alpha_qza in alpha_qza_files:

    # Extract the alpha diversity metric name from the filename
    alpha_metric = os.path.basename(alpha_qza).replace("_vector.qza", "")

    # Define export directory for this metric
    alpha_export = f"{alpha_raw_dir}/export_{alpha_metric}"

    # Export the QIIME artifact to a TSV file
    !qiime tools export --input-path {alpha_qza} --output-path {alpha_export}

    # Read the exported alpha diversity table
    df_alpha = pd.read_csv(
        f"{alpha_export}/alpha-diversity.tsv",
        sep="\t"
    )

    # Standardize the sample ID column name
    df_alpha = df_alpha.rename(
        columns={"#SampleID": "id", "Unnamed: 0": "id"}
    )

    # Rename the alpha diversity column to the metric name
    df_alpha = df_alpha.rename(
        columns={df_alpha.columns[1]: alpha_metric}
    )

    # Store the dataframe for later merging
    alpha_dfs.append(df_alpha)

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported ../data/processed/diversity/depth-14000_no-filter/alpha/metrics/faith_pd_vector.qza as AlphaDiversityDirectoryFormat to directory ../data/processed/longitudinal/depth-14000_no-filter/alpha_raw/export_faith_pd
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported ../data/processed/diversity/depth-14000_no-filter/alpha/metrics/shannon_vector.qza as AlphaDiversityDirectoryFormat t

After exporting and cleaning the individual alpha diversity metrics,
we merge them into a single table.

This results in one dataframe where:
- each row represents a sample
- each column represents an alpha diversity metric
- the sample ID is used as the merge key

We then merge this table with the sample metadata to create
a single metadata table that includes all alpha diversity values.

In [8]:
# Start with the first alpha metric dataframe
alpha_merged = alpha_dfs[0].copy()

# Merge remaining alpha metrics one by one
for df in alpha_dfs[1:]:

    alpha_merged = alpha_merged.merge(
        df,
        on="id",
        how="inner"
    )

# Merge with metadata
merged_alpha = metadata.merge(alpha_merged, on="id", how="inner")

# Save final merged table
alpha_out = f"{alpha_raw_dir}/metadata_raw.tsv"
merged_alpha.to_csv(alpha_out, sep="\t", index=False)

print("Saved merged alpha metrics to:", alpha_out)

Saved merged alpha metrics to: ../data/processed/longitudinal/depth-14000_no-filter/alpha_raw/metadata_raw.tsv


Below, the merged alpha diversity table is shown.
Each row represents a sample, and the columns include the sample metadata
together with all alpha diversity metrics.

In [9]:
merged_alpha

,id,host_id,age_months,geo_location_name,delivery_mode,sex,diet_weaning,diet_milk,treatment_exposure,age_days,faith_pd,shannon,evenness,observed_features
0,SRR8118533,E000823,4.0,Finland,vaginal,male,no,bd,False,121,11.319646,3.463449,0.675231,35
1,SRR8118537,E000823,7.0,Finland,vaginal,male,yes,mixed,False,205,15.527365,4.297361,0.689916,75
2,SRR8118564,E001958,4.0,Finland,vaginal,female,yes,bd,False,129,6.253833,2.292308,0.549724,18
3,SRR8118650,E001958,7.0,Finland,vaginal,female,yes,mixed,False,223,9.001924,3.619116,0.634884,52
4,SRR8118652,E001958,10.0,Finland,vaginal,female,yes,mixed,False,319,13.652485,3.942067,0.652186,66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,SRR8116456,T026211,10.0,Estonia,vaginal,male,yes,mixed,False,299,11.374028,5.657813,0.863183,94
306,SRR8120711,T028183,7.0,Estonia,cesarean,female,yes,mixed,True,218,12.839270,4.921306,0.743994,98
307,SRR8120280,T029687,7.0,Estonia,cesarean,male,yes,NaN,True,226,11.842117,5.836889,0.836557,126
308,SRR8120284,T029922,7.0,Estonia,cesarean,male,yes,NaN,True,221,11.855160,3.990770,0.667656,63


### 1.2 Calculate first differences in alpha diversity over time

In this step, we calculate first differences for alpha diversity.
First differences describe how much alpha diversity changes between
two consecutive sampling time points within the same individual.

A first difference is calculated as:

`difference = alpha(t₂) − alpha(t₁)`

This answers the question:
“How much did this person’s microbiome diversity change since the last visit?” 

Positive values indicate an increase in alpha diversity,
negative values indicate a decrease,
and values close to zero indicate little change between visits. First differences give insight into how stable or dynamic
the microbiome is over time. Although these results are interesting,
they are not used further in this analysis pipeline.
A more detailed analysis of alpha diversity first differences
is left for future work.

In [10]:
# Choose the alpha diversity metric
chosen_alpha_metric = "shannon"

# Define output path for first differences
first_differences_qza = f"{alpha_fd_dir}/{chosen_alpha_metric}_firstdiff.qza"

# Compute first differences using QIIME
!qiime longitudinal first-differences \
  --p-metric {chosen_alpha_metric} \
  --m-metadata-file {alpha_out} \
  --p-state-column {time_column} \
  --p-individual-id-column host_id \
  --p-replicate-handling random \
  --o-first-differences {first_differences_qza} \
  --verbose

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/conda/lib/python3.10/site-packages/q2_longitudinal/_utilities.py:446: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata = metadata.apply(lambda x: pd.to_numeric(x, errors='ignore'))
Saved SampleData[FirstDifferences] to: ../data/processed/longitudinal/depth-14000_no-filter/alpha_firstdiff/shannon_firstdiff.qza


We export the first-difference QZA file and merge it with the metadata. This creates a second metadata table specifically for analyzing
the *change* in diversity instead of the raw values.

### 1.3 Compute first distances in beta diversity over time

While alpha diversity first differences describe changes in overall diversity,
beta diversity focuses on changes in community composition.
In this step, we therefore calculate first distances in beta diversity.

First distances quantify how much the microbial community composition
changes between two consecutive sampling time points within the same individual.

A first distance is calculated as:

`distance = beta(sample(t₂), sample(t₁))`

This answers a complementary question to alpha diversity:
“How much did this person’s microbiome composition change since the last visit,
regardless of whether diversity increased or decreased?”

Larger values indicate stronger shifts in community composition,
while smaller values suggest more stable microbiomes between visits.
First distances provide insight into temporal changes in composition that may not
be captured by alpha diversity alone. Although these results are also informative, beta diversity first distances are not
used further in this analysis pipeline. A more detailed investigation of compositional change over time
is left for future work.



In [11]:
chosen_beta_metric = "bray_curtis"
beta_dir = f"{diversity_data_dir}/{run}/beta/metrics"
beta_dm_qza = f"{beta_dir}/{chosen_beta_metric}_distance_matrix.qza"

first_distances_qza = f"{beta_fd_dir}/{chosen_beta_metric}_firstdist.qza"

!qiime longitudinal first-distances \
  --i-distance-matrix {beta_dm_qza} \
  --m-metadata-file {alpha_out} \
  --p-state-column {time_column} \
  --p-individual-id-column host_id \
  --p-replicate-handling random \
  --o-first-distances {first_distances_qza} \
  --verbose

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/conda/lib/python3.10/site-packages/q2_longitudinal/_utilities.py:446: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata = metadata.apply(lambda x: pd.to_numeric(x, errors='ignore'))
Saved SampleData[FirstDifferences] to: ../data/processed/longitudinal/depth-14000_no-filter/beta_firstdist/bray_curtis_firstdist.qza


## 2.  Longitudinal exploration of diversity patterns  

Volatility plots are used to visualize how a metric changes over time for each
child in the study. In this section, we apply this approach to alpha diversity
metrics, such as Shannon diversity. These plots provide a clear overview of individual trajectories and the overall
pattern across age. By looking at all children together, we can see whether
alpha diversity tends to increase, decrease, or remain relatively stable as
children grow.

Volatility plots also highlight differences between individuals. Some children
show smooth, gradual changes in alpha diversity, while others exhibit more
variation over time. This helps to illustrate the heterogeneity in microbiome
development. Overall, these visualizations help us understand the structure of the data
before performing statistical modeling. They also give an indication of whether
a linear mixed-effects model is appropriate, by showing whether there appears
to be a consistent time-related trend that can be captured by such a model [[Chen, 2022](https://pmc.ncbi.nlm.nih.gov/articles/PMC9285460/)]..


In [12]:
# Output path for volatility plot of raw alpha diversity
vol_raw_out = f"{volatility_raw_dir}/volatility_raw.qzv"

# Create volatility plot for raw alpha diversity values
!qiime longitudinal volatility \
  --m-metadata-file {alpha_out} \
  --p-default-metric {alpha_metric} \
  --p-state-column {time_column} \
  --p-individual-id-column host_id \
  --o-visualization {vol_raw_out}

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/longitudinal/depth-14000_no-filter/volatility_raw/volatility_raw.qzv


In [13]:
Visualization.load(vol_raw_out)

<visualization: Visualization uuid: d6d7324d-70b9-487c-980d-f9b074c1700e>

Overall, alpha diversity metrics tend to increase with age across individuals, although there is considerable variability between subjects. This increasing pattern is visible for nearly all groups shown in the raw data. Differences between groups appear relatively small compared to the overall age-related trend, suggesting that age may be an important factor shaping alpha diversity. However, based on these descriptive plots alone, it is not possible to determine whether group effects are statistically meaningful. These observations will therefore be explored more formally in the next sections of the analysis.



## 3. Longitudinal modeling of alpha diversity  

In this section, we model alpha diversity longitudinally using
linear mixed-effects models. Linear mixed-effects models are well suited for longitudinal data
because they account for repeated measurements within individuals.
This allows us to assess the association between alpha diversity
and explanatory variables while controlling for within-subject correlation.


### 3.1 Define function for linear mixed-effect modeling 

Before fitting the models, we define a set of helper functions.
These functions automate running multiple linear mixed-effects models,
checking model convergence, and extracting the relevant results
from QIIME output files. Defining these functions keeps the analysis reproducible
and avoids repeating similar code for each model.


In [14]:
def check_lme_convergence(qzv_path):
    """
    Check whether a QIIME2 linear mixed-effects model converged.

    The function reads the model summary inside the .qzv file
    and returns 'Yes', 'No', or an error message.
    """
    try:
        with zipfile.ZipFile(qzv_path, "r") as z:
            summary_path = None
            for name in z.namelist():
                if name.endswith("data/model_summary.tsv"):
                    summary_path = name
                    break

            if summary_path is None:
                return "Summary not found"

            with z.open(summary_path) as f:
                df = pd.read_csv(f, sep="\t")

            df.columns = [c.strip() for c in df.columns]

            row = df[df[df.columns[0]].str.contains("Converged", na=False)]

            if row.empty:
                return "Convergence row missing"

            return str(row.iloc[0, 1]).strip()

    except Exception as e:
        return f"Error: {e}"

This function runs linear mixed-effects models for one outcome variable
(e.g. an alpha diversity metric) against multiple metadata variables. Each model is fitted separately, and convergence is checked automatically.


In [15]:
def run_lme_for_column(column_to_test, metadata_path, output_dir, variables):
    """
    Run QIIME2 linear mixed-effects models for one outcome variable
    against multiple metadata predictors, automatically dropping
    samples with missing values for each predictor.
    """

    print(f" RUNNING LME MODELS FOR: {column_to_test}")

    results = []

    # Load metadata once
    metadata = pd.read_csv(metadata_path, sep="\t")

    for var in variables:
        print(f"Running LME for predictor: {var}")

        # Drop samples with missing values for this variable
        meta_filtered = metadata.dropna(subset=[var])

        # Skip if too few samples remain
        if meta_filtered.shape[0] < 5:
            print(f" Skipping {var}: too few samples after filtering")
            results.append({
                "column": column_to_test,
                "variable": var,
                "converged": "Too few samples",
                "qzv_path": None
            })
            continue

        # Save temporary metadata file
        tmp_meta_path = f"{output_dir}/tmp_metadata_{column_to_test}_{var}.tsv"
        meta_filtered.to_csv(tmp_meta_path, sep="\t", index=False)

        out_qzv = f"{output_dir}/LME_{column_to_test}_{var}.qzv"

        cmd = f"""
        qiime longitudinal linear-mixed-effects \
          --m-metadata-file {tmp_meta_path} \
          --p-metric {column_to_test} \
          --p-state-column age_months \
          --p-individual-id-column host_id \
          --p-group-columns {var} \
          --p-random-effects age_months \
          --o-visualization {out_qzv}
        """

        process = subprocess.run(
            cmd,
            shell=True,
            capture_output=True,
            text=True
        )

        if process.returncode != 0:
            print(f" Model failed for {var}")
            results.append({
                "column": column_to_test,
                "variable": var,
                "converged": "Error",
                "qzv_path": out_qzv
            })
            continue

        converged = check_lme_convergence(out_qzv)
        print(f"  → Converged? {converged}")

        results.append({
            "column": column_to_test,
            "variable": var,
            "converged": converged,
            "qzv_path": out_qzv
        })

    return pd.DataFrame(results)

The following helper functions are used to extract model results
from QIIME `.qzv` files and format them for easier interpretation and add stars to see whether results are significant quickly.


In [16]:
def extract_model_results(qzv_path):
    """
    Extract the model_results.tsv table from a QIIME2 LME .qzv file.
    """
    try:
        with zipfile.ZipFile(qzv_path, "r") as z:
            for name in z.namelist():
                if name.endswith("data/model_results.tsv"):
                    with z.open(name) as f:
                        return pd.read_csv(f, sep="\t")
    except Exception as e:
        print(f"Error reading {qzv_path}: {e}")

    return None

# Add significance stars based on p-value
def add_sig_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return ""

This function extracts fixed-effect results from converged models
and removes random-effect rows, which are not used for interpretation.

In [17]:
# Extract fixed-effects results for one modeled column
def extract_lme_results_for_column(results_df):
    """
    Extract fixed-effects results from converged QIIME2 LME models
    for a single modeled column.
    """
    grouped_results = {}

    # Only process converged models
    converged_df = results_df[results_df["converged"] == "Yes"]

    for _, row in converged_df.iterrows():
        var = row["variable"]
        qzv = row["qzv_path"]

        df = extract_model_results(qzv)
        if df is None:
            continue

        # Rename effect column
        df = df.rename(columns={"Unnamed: 0": "effect"})

        # Remove random-effect rows (which have missing z and p-values)
        df = df[df["z"].notna()].copy()

        # Add p-values and significance stars
        df["p"] = df["P>|z|"].astype(float)
        df["significance"] = df["p"].apply(add_sig_stars)

        # Store which metadata variable was tested
        df["tested_variable"] = var

        grouped_results[var] = df

    return grouped_results

### 3.2 Fit linear mixed-effects models for alpha diversity

In this step, we fit linear mixed-effects models [[QIIME 2 Documentation, 2022](https://docs.qiime2.org/2022.8/tutorials/longitudinal/#linear-mixed-effect-models)] to the raw alpha diversity
metrics to test whether their trajectories over time differ depending on
specific metadata variables, such as geographic location, delivery mode, sex,
or treatment exposure. For each alpha diversity metric, separate models are fitted for each metadata
variable. The results are returned as QIIME2 visualizations, which summarize the
estimated effects and their statistical support.

In [18]:
# Metadata variables to test as fixed effects
variables = [
    "diet_weaning",
    "diet_milk",
    "geo_location_name",
    "delivery_mode",
    "sex",
    "treatment_exposure",
]

In [19]:
# Run LME models for Faith's Phylogenetic Diversity
results_faith_pd = run_lme_for_column(
    column_to_test="faith_pd",
    metadata_path=alpha_out,
    output_dir=alpha_lme_results_dir,
    variables=variables
)

 RUNNING LME MODELS FOR: faith_pd
Running LME for predictor: diet_weaning
  → Converged? No
Running LME for predictor: diet_milk
  → Converged? No
Running LME for predictor: geo_location_name
  → Converged? No
Running LME for predictor: delivery_mode
  → Converged? No
Running LME for predictor: sex
  → Converged? Yes
Running LME for predictor: treatment_exposure
  → Converged? No


In [20]:
# Run LME models for Shannon diversity
results_shannon = run_lme_for_column(
    column_to_test="shannon",
    metadata_path=alpha_out,
    output_dir=alpha_lme_results_dir,
    variables=variables
)

 RUNNING LME MODELS FOR: shannon
Running LME for predictor: diet_weaning
  → Converged? Yes
Running LME for predictor: diet_milk
  → Model failed for diet_milk
Running LME for predictor: geo_location_name
  → Converged? Yes
Running LME for predictor: delivery_mode
  → Converged? No
Running LME for predictor: sex
  → Converged? Yes
Running LME for predictor: treatment_exposure
  → Converged? Yes


In [21]:
# Run LME models for Pielou's evenness
results_evenness = run_lme_for_column(
    column_to_test="evenness",
    metadata_path=alpha_out,
    output_dir=alpha_lme_results_dir,
    variables=variables
)

 RUNNING LME MODELS FOR: evenness
Running LME for predictor: diet_weaning
  → Converged? No
Running LME for predictor: diet_milk
  → Converged? Yes
Running LME for predictor: geo_location_name
  → Converged? No
Running LME for predictor: delivery_mode
  → Converged? Yes
Running LME for predictor: sex
  → Converged? No
Running LME for predictor: treatment_exposure
  → Converged? Yes


### 3.3 Summarize and interpret model results  

In this section, we extract and inspect the results of the fitted
linear mixed-effects models. For each alpha diversity metric, we focus on the fixed effects of the
tested metadata variables. Only models that successfully converged
are included. The tables below report the estimated effect sizes,
standard errors, test statistics, and p-values, which together provide
an overview of which variables show evidence of an association with
alpha diversity over time.

In [22]:
# Run extraction for each modeled column
faith_pd_results_grouped = extract_lme_results_for_column(results_faith_pd)
shannon_results_grouped = extract_lme_results_for_column(results_shannon)
evenness_results_grouped = extract_lme_results_for_column(results_evenness)

print("Extracted grouped LME tables for all modeled columns.")

Extracted grouped LME tables for all modeled columns.


In [23]:
# Display results for Faith's Phylogenetic Diversity
for var, df in faith_pd_results_grouped.items():
    print(f"\Faith PD – effect of {var}")
    display(df[["effect", "Coef.", "Std.Err.", "z", "p", "significance"]])


=== Faith PD – effect of sex ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,4.699,0.734,6.405,0.000,***
1,sex[T.male],3.305,1.071,3.086,0.002,**
2,age_months,0.859,0.094,9.091,0.000,***
3,age_months:sex[T.male],-0.449,0.134,-3.350,0.001,**


In [24]:
# Display results for Shannon diversity
for var, df in shannon_results_grouped.items():
    print(f"\n Shannon – effect of {var}")
    display(df[["effect", "Coef.", "Std.Err.", "z", "p", "significance"]])


=== Shannon – effect of diet_weaning ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,1.785,0.663,2.694,0.007,**
1,diet_weaning[T.yes],0.729,0.684,1.067,0.286,
2,age_months,0.324,0.152,2.123,0.034,*
3,age_months:diet_weaning[T.yes],-0.187,0.154,-1.214,0.225,



=== Shannon – effect of geo_location_name ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,2.090,0.323,6.468,0.000,***
1,geo_location_name[T.Finland],0.524,0.371,1.415,0.157,
2,geo_location_name[T.Russia],0.564,0.521,1.083,0.279,
3,age_months,0.245,0.041,5.920,0.000,***
4,age_months:geo_location_name[T.Finland],-0.123,0.048,-2.582,0.010,*
5,age_months:geo_location_name[T.Russia],-0.166,0.064,-2.594,0.009,**



=== Shannon – effect of sex ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,2.522,0.230,10.963,0.000,***
1,sex[T.male],0.001,0.300,0.003,0.998,
2,age_months,0.146,0.030,4.858,0.000,***
3,age_months:sex[T.male],-0.020,0.039,-0.513,0.608,



=== Shannon – effect of treatment_exposure ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,2.474,0.233,10.609,0.000,***
1,treatment_exposure[T.True],0.082,0.302,0.271,0.786,
2,age_months,0.137,0.030,4.610,0.000,***
3,age_months:treatment_exposure[T.True],-0.005,0.039,-0.120,0.904,


In [25]:
# Display results for Evenness
for var, df in evenness_results_grouped.items():
    print(f"\nEvenness – effect of {var}")
    display(df[["effect", "Coef.", "Std.Err.", "z", "p", "significance"]])


=== Evenness – effect of diet_milk ===


,effect,Coef.,Std.Err.,z,p,significance



=== Evenness – effect of delivery_mode ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,0.671,0.087,7.711,0.000,***
1,delivery_mode[T.vaginal],-0.104,0.091,-1.153,0.249,
2,age_months,-0.003,0.011,-0.228,0.819,
3,age_months:delivery_mode[T.vaginal],0.011,0.012,0.950,0.342,



=== Evenness – effect of treatment_exposure ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,0.557,0.039,14.441,0.000,***
1,treatment_exposure[T.True],0.031,0.050,0.620,0.535,
2,age_months,0.008,0.005,1.794,0.073,
3,age_months:treatment_exposure[T.True],-0.001,0.006,-0.154,0.878,


To interpret the results, it is useful to distinguish between overall differences between metadata groups and changes in alpha diversity over time.

Differences between metadata groups:
- Across all alpha diversity metrics, there is little evidence that children differ
in their overall alpha diversity levels based solely on metadata variables such as
delivery mode, diet weaning status, treatment exposure, or geographic location.
The main effects of these variables are generally not significant, indicating that
the groups do not differ strongly in their average alpha diversity values.
This suggests that factors such as delivery mode or treatment exposure do not, by
themselves, lead to consistently higher or lower alpha diversity across the
observed age range.

Changes in alpha diversity over time:
- In contrast, age shows a consistent and significant association with alpha
diversity for both Faith’s Phylogenetic Diversity and Shannon diversity.
These results indicate that alpha diversity increases as children grow older,
supporting the idea of progressive microbiome development during early life.
For Shannon diversity, there is some evidence that the rate of increase differs
slightly by geographic location, as reflected by significant age × location
interaction terms. This suggests that while alpha diversity increases with age in
all groups, the speed of this increase may vary modestly between countries.
For evenness, both age or other metadata variables do not show significant effects,
indicating that the relative distribution of abundances across taxa remains
largely stable over time.


Overall, these findings indicate that age is the primary driver of changes in
alpha diversity, while the tested metadata variables do not show strong or
consistent associations with alpha diversity levels themselves. Any group-related
differences appear to be subtle and mainly expressed through differences in
developmental trajectories rather than clear separation between groups.

## 4. Longitudinal modeling of taxa associated with microbiome development

In this section, we shift the focus from overall diversity measures to individual
microbial taxa. While alpha and beta diversity describe global patterns in the
microbiome, they do not reveal which specific taxa are responsible for these
changes over time. This analysis is based on a similar approach described in [[Bokulich et al., 2018](https://doi.org/10.1128/msystems.00219-18)].

Importantly, not all taxa show the same temporal behavior during early-life
microbiome development. Many taxa remain relatively stable across time, whereas
a smaller subset changes more strongly as infants grow, transition in diet, or
are exposed to environmental factors. Focusing on all taxa simultaneously would
therefore include many features that show little or no meaningful variation. To address this, we first use feature volatility as an exploratory step.
Feature volatility applies a machine-learning approach to identify taxa whose
abundances are most informative for predicting age. This allows us to highlight
features that show strong temporal dynamics, without making assumptions about
the direction or cause of these changes.

The taxa identified by feature volatility are then used as candidates for
longitudinal modeling. In the second step, we fit linear mixed-effects models to
these selected taxa to formally assess how their abundances change over time and
whether these patterns differ across metadata variables such as diet or
treatment exposure. This two-step approach helps reduce noise, improves
interpretability, and allows us to focus on taxa that are most likely to be
biologically relevant for microbiome development.


### 4.1 Identify age- or diet-associated taxa using feature volatility 

Here, we use feature volatility to identify taxa that show strong temporal
patterns in the data. Feature volatility fits a random forest model to predict
the time variable based on microbial abundances. The model ranks taxa according to how informative they are for predicting age.
Taxa with higher importance scores show stronger associations with temporal
development, while taxa with low importance contribute little to the prediction.

In [26]:
# Note: Running feature volatility can take some time.
# This is a good moment for another short coffee break 

# Define output directory for feature volatility results
fv_outdir = f"{analysis_dir}/feature_volatility_raw"

# Remove existing output directory if it already exists
if os.path.exists(fv_outdir):
    shutil.rmtree(fv_outdir)
    print(f"Removed existing directory: {fv_outdir}")
else:
    print(f"No existing directory found: {fv_outdir}")

# Run feature volatility analysis
cmd = f"""
qiime longitudinal feature-volatility \
  --i-table {diversity_data_dir}/{run}/rarefied_table.qza \
  --m-metadata-file {alpha_out} \
  --p-state-column {time_column} \
  --p-individual-id-column host_id \
  --output-dir {analysis_dir}/feature_volatility_raw \
  --verbose
"""

print(cmd)
!{cmd}

Removed existing directory: ../data/processed/longitudinal/depth-14000_no-filter/feature_volatility_raw

qiime longitudinal feature-volatility   --i-table ../data/processed/diversity/depth-14000_no-filter/rarefied_table.qza   --m-metadata-file ../data/processed/longitudinal/depth-14000_no-filter/alpha_raw/metadata_raw.tsv   --p-state-column age_months   --p-individual-id-column host_id   --output-dir ../data/processed/longitudinal/depth-14000_no-filter/feature_volatility_raw   --verbose

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/conda/lib/python3.10/site-packages/q2_sample_classifier/_transformer.py:69: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric wi

The next visualization shows how the most important microbial features change across the age range.
Each curve represents one taxon, and its shape indicates how its abundance shifts over time.

In [27]:
Visualization.load(f"{analysis_dir}/feature_volatility_raw/volatility_plot.qzv")

<visualization: Visualization uuid: e48e1090-2e49-40fd-af3c-ad3e883d0dda>

The next visualization summarizes how well the Random Forest regressor predicts infant age.
It includes:

- Correlation between predicted and actual age  
- R² value (variance explained)  
- Mean squared error (MSE)  
- Slope and intercept of the regression line  

Together, these indicate how strongly the microbiome reflects developmental age.

In [28]:
Visualization.load(f"{analysis_dir}/feature_volatility_raw/accuracy_results.qzv")

<visualization: Visualization uuid: 6d352025-519d-489c-b7dc-89a38cdbbfdd>

In this step, we take the feature importance values from the feature-volatility analysis and match
them with their taxonomy. The Random Forest model tells us which microbial features (ASVs) are most
helpful for predicting a child’s age, but the output only shows ASV IDs, which aren’t meaningful on
their own. By merging these IDs with the taxonomy file, we can finally see what these features
actually are, for example, which phylum, family, or genus they belong to.

This is important because it lets us interpret the model in a biological way. Instead of just
knowing that “Feature_123” is important, we can see whether it’s a Bacteroides, a Blautia, or
something else that changes with age. After merging, we apply a small cutoff (0.02 by default) to
focus only on the ASVs that the model considers genuinely important. The final printed list shows
the feature ID, its importance score, and its full taxonomy, making it easier to understand which
microbes are actually driving age-related patterns in the dataset.

In [29]:
# Export feature importance from the raw feature-volatility output
!qiime tools export \
  --input-path {analysis_dir}/feature_volatility_raw/feature_importance.qza \
  --output-path {analysis_dir}/feature_volatility_raw/importance_export

# Export taxonomy so we can annotate the features
!qiime tools export \
  --input-path {taxonomy_data_dir}/taxonomy.qza \
  --output-path {taxonomy_data_dir}/taxonomy_export

# Load importance values + taxonomy
importance_path = f"{analysis_dir}/feature_volatility_raw/importance_export/importance.tsv"
taxonomy_path = f"{taxonomy_data_dir}/taxonomy_export/taxonomy.tsv"

df_imp = pd.read_csv(importance_path, sep="\t")
df_tax = pd.read_csv(taxonomy_path, sep="\t")

# Merge importance + taxonomy
merged = df_imp.merge(df_tax, left_on="feature", right_on="Feature ID", how="left")

# Filter by importance threshold
threshold = 0.02 
top_features = (
    merged[merged["importance"] >= threshold]
    .sort_values("importance", ascending=False)
)

# Print top features with taxonomy
for _, row in top_features.iterrows():
    print("Feature ID:", row["feature"])
    print("Importance:", round(row["importance"], 4))
    print("Taxonomy: ", row["Taxon"] if "Taxon" in row else row.get("taxonomy", "N/A"))
    print("-" * 80)

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported ../data/processed/longitudinal/depth-14000_no-filter/feature_volatility_raw/feature_importance.qza as ImportanceDirectoryFormat to directory ../data/processed/longitudinal/depth-14000_no-filter/feature_volatility_raw/importance_export
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported ../data/processed/taxonomy/taxonomy.qza as TSVTaxonomyDirectoryFormat to directory ../data/

The feature volatility analysis identifies a small number of microbial features
that are most informative for predicting age. Among these, Blautia and
Thomasclavelia have the highest importance scores, indicating that their
abundances change more strongly with age compared to other taxa in the dataset.

The sharp drop in importance scores after the top features suggests that
age-related microbiome development is not driven by widespread changes across
many taxa, but rather by pronounced shifts in a limited subset of microbial
features. Other taxa, such as Agathobacter, Flavonifractor, and members of the
[Clostridium] innocuum group, contribute to the model to a lesser extent.

It is important to note that feature importance reflects how useful a taxon is
for predicting age within the random forest model, not statistical significance
or causal effects. These results therefore serve as an exploratory ranking of
taxa based on temporal dynamics. Based on this ranking, Blautia and
Thomasclavelia were selected for further longitudinal modeling.

### 4.2 Fit linear mixed-effects models for selected taxa  

Based on the feature volatility analysis, we identified *Blautia* and
*Thomasclavelia* as taxa showing strong temporal patterns. In this step, we
examine these taxa in more detail using linear mixed-effects models. For each selected taxon, we fit a linear mixed-effects model to assess how its
abundance changes over time and whether this pattern differs across metadata
variables. This approach allows us to formally test longitudinal associations
while accounting for repeated measurements within individuals.

To fit linear mixed-effects models for the selected taxa, we first need to
prepare a metadata table that includes their abundances per sample. We export the rarefied feature table, extract the ASVs identified in the feature
volatility analysis, map them to their corresponding genera, and merge the
resulting abundance data with the sample metadata. This produces a single table
that can be used directly for longitudinal modeling.

In [30]:
# Export the rarefied feature table from QIIME
!qiime tools export \
  --input-path {diversity_data_dir}/{run}/rarefied_table.qza \
  --output-path {analysis_dir}/exported_rarefied_table

# Convert BIOM table to TSV format
!biom convert \
  -i {analysis_dir}/exported_rarefied_table/feature-table.biom \
  -o {analysis_dir}/exported_rarefied_table/feature-table.tsv \
  --to-tsv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported ../data/processed/diversity/depth-14000_no-filter/rarefied_table.qza as BIOMV210DirFmt to directory ../data/processed/longitudinal/depth-14000_no-filter/exported_rarefied_table


In [31]:
# Load the exported rarefied feature table
feature_table_taxa = pd.read_csv(
    f"{analysis_dir}/exported_rarefied_table/feature-table.tsv",
    sep="\t",
    skiprows=1
)

# Rename feature ID column and set it as index
feature_table_taxa = feature_table_taxa.rename(
    columns={"#OTU ID": "feature_id"}
)
feature_table_taxa = feature_table_taxa.set_index("feature_id")

In [32]:
# ASVs selected from feature volatility analysis
# Mapped to genus level for easier interpretation
important_taxa = {
    "a663a01aeb4dbb88bbe0a9840e8f64dc": "Thomasclavelia",
    "c6c3ab4e828fb40d6e05967b7aac9338": "Blautia"
}

In [33]:
# Filter the feature table to selected ASVs
filtered_taxa_table = feature_table_taxa.loc[important_taxa.keys()]

# Transpose so rows correspond to samples
filtered_taxa_table_T = filtered_taxa_table.T
filtered_taxa_table_T.index.name = "id"

# Rename ASV columns to genus names
filtered_taxa_table_T = filtered_taxa_table_T.rename(columns=important_taxa)

# Merge selected taxa abundances with metadata
merged_taxa_metadata = metadata.merge(
    filtered_taxa_table_T,
    on="id",
    how="inner"
)

print("Merged metadata with selected genera:")
display(merged_taxa_metadata.head())

# Save combined metadata table for taxa LME analysis
lme_meta_dir = f"{analysis_dir}/taxa_LME"
os.makedirs(lme_meta_dir, exist_ok=True)

main_taxa_meta_path = f"{lme_meta_dir}/metadata_main_features.tsv"
merged_taxa_metadata.to_csv(main_taxa_meta_path, sep="\t", index=False)

print(f"Saved combined metadata file to:\n  {main_taxa_meta_path}")

Merged metadata with selected genera:


,id,host_id,age_months,geo_location_name,delivery_mode,sex,diet_weaning,diet_milk,treatment_exposure,age_days,Thomasclavelia,Blautia
0,SRR8118533,E000823,4.0,Finland,vaginal,male,no,bd,False,121,101.0,0.0
1,SRR8118537,E000823,7.0,Finland,vaginal,male,yes,mixed,False,205,139.0,553.0
2,SRR8118564,E001958,4.0,Finland,vaginal,female,yes,bd,False,129,0.0,0.0
3,SRR8118650,E001958,7.0,Finland,vaginal,female,yes,mixed,False,223,529.0,712.0
4,SRR8118652,E001958,10.0,Finland,vaginal,female,yes,mixed,False,319,23.0,300.0


Saved combined metadata file to:
  ../data/processed/longitudinal/depth-14000_no-filter/taxa_LME/metadata_main_features.tsv


In [34]:
# Metadata variables to test as fixed effects
variables = [
    "diet_weaning",
    "diet_milk",
    "geo_location_name",
    "delivery_mode",
    "sex",
    "treatment_exposure",
]

# Thomasclavelia
results_thomas = run_lme_for_column(
    column_to_test="Thomasclavelia",
    metadata_path=main_taxa_meta_path,
    output_dir=lme_meta_dir,
    variables=variables
)

# Blautia
results_blautia = run_lme_for_column(
    column_to_test="Blautia",
    metadata_path=main_taxa_meta_path,
    output_dir=lme_meta_dir,
    variables=variables
)

 RUNNING LME MODELS FOR: Thomasclavelia
Running LME for predictor: diet_weaning
  → Converged? Yes
Running LME for predictor: diet_milk
  → Converged? No
Running LME for predictor: geo_location_name
  → Converged? Yes
Running LME for predictor: delivery_mode
  → Converged? Yes
Running LME for predictor: sex
  → Converged? No
Running LME for predictor: treatment_exposure
  → Converged? No
 RUNNING LME MODELS FOR: Blautia
Running LME for predictor: diet_weaning
  → Converged? No
Running LME for predictor: diet_milk
  → Converged? No
Running LME for predictor: geo_location_name
  → Converged? Yes
Running LME for predictor: delivery_mode
  → Converged? Yes
Running LME for predictor: sex
  → Converged? Yes
Running LME for predictor: treatment_exposure
  → Converged? Yes


### 4.3 Summarize and interpret model results   

Next, we fit linear mixed-effects models for the selected taxa,
*Thomasclavelia* and *Blautia*. For each taxon, separate models are fitted to assess whether its
abundance changes over time in different ways across the tested
metadata variables.

In [35]:
# Metadata variables to test as fixed effects
variables = [
    "geo_location_name",
    "delivery_mode",
    "sex",
    "treatment_exposure"
]

# Run LME models for Thomasclavelia
results_thomas = run_lme_for_column(
    column_to_test="Thomasclavelia",
    metadata_path=main_taxa_meta_path,
    output_dir=lme_meta_dir,
    variables=variables
)

# Run LME models for Blautia
results_blautia = run_lme_for_column(
    column_to_test="Blautia",
    metadata_path=main_taxa_meta_path,
    output_dir=lme_meta_dir,
    variables=variables
)

 RUNNING LME MODELS FOR: Thomasclavelia
Running LME for predictor: geo_location_name
  → Converged? Yes
Running LME for predictor: delivery_mode
  → Converged? Yes
Running LME for predictor: sex
  → Converged? No
Running LME for predictor: treatment_exposure
  → Converged? No
 RUNNING LME MODELS FOR: Blautia
Running LME for predictor: geo_location_name
  → Converged? Yes
Running LME for predictor: delivery_mode
  → Converged? Yes
Running LME for predictor: sex
  → Converged? Yes
Running LME for predictor: treatment_exposure
  → Converged? Yes


Below we extract the results from the LME models for the two different taxa.

In [36]:
# Run extraction for each modeled column
thomas_results_grouped = extract_lme_results_for_column(results_thomas)
blautia_results_grouped = extract_lme_results_for_column(results_blautia)

print("Extracted grouped LME tables for all modeled columns.")

Extracted grouped LME tables for all modeled columns.


Below, we summarize the fixed-effect results of the linear mixed-effects models
for the selected taxa. For each taxon, the tables report the estimated effects of
the tested metadata variables, along with their standard errors and p-values.
As before, only results from models that successfully converged are shown.

In [37]:
# Display results for Thomasclavelia
for var, df in thomas_results_grouped.items():
    print(f"\n=== Thomasclavelia – effect of {var} ===")
    display(df[["effect", "Coef.", "Std.Err.", "z", "p", "significance"]])


=== Thomasclavelia – effect of geo_location_name ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,544.540,147.290,3.697,0.000,***
1,geo_location_name[T.Finland],-467.045,171.442,-2.724,0.006,**
2,geo_location_name[T.Russia],-512.330,232.547,-2.203,0.028,*
3,age_months,-49.840,18.302,-2.723,0.006,**
4,age_months:geo_location_name[T.Finland],53.315,21.349,2.497,0.013,*
5,age_months:geo_location_name[T.Russia],64.349,28.192,2.283,0.022,*



=== Thomasclavelia – effect of delivery_mode ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,88.097,245.893,0.358,0.720,
1,delivery_mode[T.vaginal],105.433,257.015,0.410,0.682,
2,age_months,1.700,30.576,0.056,0.956,
3,age_months:delivery_mode[T.vaginal],-9.472,31.819,-0.298,0.766,


In [38]:
# Display results for Blautia
for var, df in blautia_results_grouped.items():
    print(f"\n=== Blautia – effect of {var} ===")
    display(df[["effect", "Coef.", "Std.Err.", "z", "p", "significance"]])


=== Blautia – effect of geo_location_name ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,-122.538,114.554,-1.070,0.285,
1,geo_location_name[T.Finland],-19.293,132.191,-0.146,0.884,
2,geo_location_name[T.Russia],195.560,184.167,1.062,0.288,
3,age_months,34.531,16.634,2.076,0.038,*
4,age_months:geo_location_name[T.Finland],10.552,19.439,0.543,0.587,
5,age_months:geo_location_name[T.Russia],-29.953,25.233,-1.187,0.235,



=== Blautia – effect of delivery_mode ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,-234.531,192.561,-1.218,0.223,
1,delivery_mode[T.vaginal],144.676,199.767,0.724,0.469,
2,age_months,54.776,28.335,1.933,0.053,
3,age_months:delivery_mode[T.vaginal],-22.604,29.445,-0.768,0.443,



=== Blautia – effect of sex ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,-38.411,81.351,-0.472,0.637,
1,sex[T.male],-105.703,106.318,-0.994,0.320,
2,age_months,25.358,12.050,2.104,0.035,*
3,age_months:sex[T.male],14.496,15.727,0.922,0.357,



=== Blautia – effect of treatment_exposure ===


,effect,Coef.,Std.Err.,z,p,significance
0,Intercept,-100.812,82.592,-1.221,0.222,
1,treatment_exposure[T.True],-4.278,107.798,-0.040,0.968,
2,age_months,29.872,11.766,2.539,0.011,*
3,age_months:treatment_exposure[T.True],8.309,15.746,0.528,0.598,


The linear mixed-effects modeling results for the selected taxa show that
model convergence was limited, which strongly constrains the conclusions that
can be drawn.

For Thomasclavelia, none of the fitted models converged for any of the tested
metadata variables. This means that the available data do not support stable
estimation of longitudinal effects for this taxon within the current modeling
framework. As a result, no formal conclusions can be drawn about how
Thomasclavelia abundance relates to age or other metadata variables based on
these models.

For Blautia, only the model including geographic location converged.
In this model, age shows a significant positive association with Blautia
abundance, indicating that Blautia tends to increase with age across
individuals. However, the main effects of geographic location are not significant,
and the age × location interaction terms are also not significant. This suggests
that while Blautia abundance increases over time, this increase does not differ
substantially between geographic locations.

Overall, these results indicate that age-related changes in specific taxa are
more consistently detectable than differences between metadata groups.
At the same time, the frequent lack of model convergence highlights the
challenges of fitting longitudinal models at the individual taxon level, likely
due to sparsity, zero inflation, or limited sample sizes. Consequently, these
taxa-level LME results should be interpreted cautiously and primarily as
exploratory.

<div style="border: 2px solid #1f77b4; padding: 10px; border-radius: 6px; background-color: #eaf3fb;">
<b>Answers to the research questions:</b>

- How does alpha diversity (e.g., richness, Shannon diversity) vary across metadata groups, including age, feeding pattern, delivery mode, treatment, sex, and geographic location?
  - Alpha diversity increases with age across individuals.
  - No strong or consistent differences in overall alpha diversity levels were observed between metadata groups such as delivery mode, feeding pattern, treatment exposure, sex, or geographic location.

- Do metadata factors such as feeding pattern, delivery mode, treatment, sex, and geographic location influence the <i>trajectory</i> of alpha diversity change over time (i.e. time × factor effects)?
  - Age is the main driver of changes in alpha diversity over time.
  - Limited evidence suggests that geographic location may slightly modify the rate of increase in Shannon diversity, but most metadata factors do not significantly alter alpha diversity trajectories.

- Which microbial features (ASVs or taxa) show the strongest age-related changes, and how do their longitudinal patterns differ across metadata factors such as geographic location, delivery mode, or treatment exposure?
  - Feature volatility identified <i>Blautia</i> and <i>Thomasclavelia</i> as taxa with the strongest age-related patterns.
  - Formal longitudinal modeling provided evidence for an age-related increase in <i>Blautia</i>, but did not reveal clear differences in taxon trajectories across metadata groups.
  
</div>


# References




Bokulich, N. A., Dillon, M. R., Zhang, Y., Rideout, J. R., Bolyen, E., Li, H.,
Albert, P. S., & Caporaso, J. G. (2018).
*Longitudinal and paired-sample analyses of microbiome data*.
mSystems, 3(6), e00219-18.
https://doi.org/10.1128/msystems.00219-18

Chen, H. (2022).
*Microbiome analysis: From raw reads to ecological interpretation*.
Frontiers in Microbiology.
https://pmc.ncbi.nlm.nih.gov/articles/PMC9285460/

National Cancer Institute, Center for Cancer Research.
*QIIME 2: Lesson 5 — Longitudinal Data Analysis*.
https://bioinformatics.ccr.cancer.gov/docs/qiime2/Lesson5/

QIIME 2 Documentation (2022).
*Longitudinal Analysis Tutorial — Linear Mixed Effects Models*.
https://docs.qiime2.org/2022.8/tutorials/longitudinal/#linear-mixed-effect-models